#### Importing Required Libraries

In [2]:
# Importing libraries for data handling.

import pandas as pd
import numpy as np


# Importing libraries for data visualization.

import plotly.express as px
import plotly.graph_objects as go


# Importing libraries for data preprocessing and model development.

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# Importing regression models for demand prediction.

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)


# Importing libraries for model evaluation.

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

#### Loading the Engineered Dataset

In [3]:
# Loading the engineered dataset for model development.

df = pd.read_csv(
    '../data/processed/engineered_data.csv'
)

In [4]:
df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,...,Is_Weekend,Inventory_to_Sales_Ratio,Inventory_Gap,Price_Difference,Price_Difference_Percentage,Promotion_Discount,Previous_Demand,Previous_Units_Sold,Rolling_7_Day_Demand,Rolling_7_Day_Sales
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,...,1,1.911765,93,-13.01,-15.175551,0,NaN,NaN,NaN,NaN
1,2022-01-02,S001,P0001,Electronics,North,93,71,0,65.63,5,...,1,1.309859,22,-8.03,-10.901439,0,115.0,102.0,NaN,NaN
2,2022-01-03,S001,P0001,Electronics,North,274,142,229,68.55,15,...,0,1.929577,132,-12.18,-15.087328,15,84.0,71.0,NaN,NaN
3,2022-01-04,S001,P0001,Electronics,North,132,42,0,61.66,10,...,0,3.142857,90,6.78,12.354227,0,132.0,142.0,NaN,NaN
4,2022-01-05,S001,P0001,Electronics,North,319,129,0,59.56,25,...,0,2.472868,190,2.22,3.871643,25,67.0,42.0,NaN,NaN


In [5]:
# Checking the dataset structure and data types.

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76000 entries, 0 to 75999
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Date                         76000 non-null  object 
 1   Store ID                     76000 non-null  object 
 2   Product ID                   76000 non-null  object 
 3   Category                     76000 non-null  object 
 4   Region                       76000 non-null  object 
 5   Inventory Level              76000 non-null  int64  
 6   Units Sold                   76000 non-null  int64  
 7   Units Ordered                76000 non-null  int64  
 8   Price                        76000 non-null  float64
 9   Discount                     76000 non-null  int64  
 10  Weather Condition            76000 non-null  object 
 11  Promotion                    76000 non-null  int64  
 12  Competitor Pricing           76000 non-null  float64
 13  Seasonality     

##### Converting the Date Column

In [6]:
# Converting the Date column into datetime format.

df['Date'] = pd.to_datetime(
    df['Date']
)

#### Sorting the Dataset by Time

In [7]:
# Sorting the dataset by date to maintain chronological order.

df = df.sort_values(
    'Date'
).reset_index(
    drop=True
)

#### Checking Missing Values

In [8]:
# Checking missing values before preparing the modeling dataset.

df.isnull().sum().sort_values(
    ascending=False
)

Rolling_7_Day_Sales            700
Rolling_7_Day_Demand           700
Previous_Units_Sold            100
Previous_Demand                100
Date                             0
Store ID                         0
Product ID                       0
Units Ordered                    0
Price                            0
Discount                         0
Weather Condition                0
Category                         0
Region                           0
Inventory Level                  0
Units Sold                       0
Epidemic                         0
Seasonality                      0
Competitor Pricing               0
Promotion                        0
Demand                           0
Year                             0
Month                            0
Week                             0
Inventory_to_Sales_Ratio         0
Is_Weekend                       0
Day_of_Week                      0
Day                              0
Promotion_Discount               0
Price_Difference_Per

#### Handling the missing values

In [9]:
# Removing rows where lag and rolling features cannot be calculated due to insufficient historical data.

df = df.dropna(
    subset=[
        'Previous_Demand',
        'Previous_Units_Sold',
        'Rolling_7_Day_Demand',
        'Rolling_7_Day_Sales'
    ]
).reset_index(
    drop=True
)

In [10]:
# Verifying that missing values have been removed from the modeling dataset.

df.isnull().sum().sort_values(
    ascending=False
)

Date                           0
Store ID                       0
Product ID                     0
Category                       0
Region                         0
Inventory Level                0
Units Sold                     0
Units Ordered                  0
Price                          0
Discount                       0
Weather Condition              0
Promotion                      0
Competitor Pricing             0
Seasonality                    0
Epidemic                       0
Demand                         0
Year                           0
Month                          0
Week                           0
Day                            0
Day_of_Week                    0
Is_Weekend                     0
Inventory_to_Sales_Ratio       0
Inventory_Gap                  0
Price_Difference               0
Price_Difference_Percentage    0
Promotion_Discount             0
Previous_Demand                0
Previous_Units_Sold            0
Rolling_7_Day_Demand           0
Rolling_7_

In [11]:
# Checking the dataset shape after removing rows with insufficient historical data.

print("Dataset Shape:", df.shape)

Dataset Shape: (75300, 31)


#### Defining Target and Features

In [12]:
# Defining Demand as the target variable for the forecasting models.
target = 'Demand'

# Defining the features that will be used to predict demand.
feature_columns = [
    'Store ID',
    'Product ID',
    'Category',
    'Region',
    'Inventory Level',
    'Units Sold',
    'Units Ordered',
    'Price',
    'Discount',
    'Weather Condition',
    'Promotion',
    'Competitor Pricing',
    'Seasonality',
    'Epidemic',
    'Year',
    'Month',
    'Week',
    'Inventory_to_Sales_Ratio',
    'Is_Weekend',
    'Day_of_Week',
    'Day',
    'Promotion_Discount',
    'Price_Difference_Percentage',
    'Price_Difference',
    'Inventory_Gap',
    'Previous_Demand',
    'Previous_Units_Sold',
    'Rolling_7_Day_Demand',
    'Rolling_7_Day_Sales'
]

# Creating the feature matrix and target variable for model development.

X = df[feature_columns]
y = df[target]

# Verifying the dimensions of the feature matrix and target variable.

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Matrix Shape: (75300, 29)
Target Shape: (75300,)


#### Checking Feature Leakage

Before training, we're checking whether any feature contains information that would not actually be available when predicting future demand.

###### Checking Feature Availability

In [13]:
# Displaying all features that are currently being considered for model development.

X.columns.tolist()

['Store ID',
 'Product ID',
 'Category',
 'Region',
 'Inventory Level',
 'Units Sold',
 'Units Ordered',
 'Price',
 'Discount',
 'Weather Condition',
 'Promotion',
 'Competitor Pricing',
 'Seasonality',
 'Epidemic',
 'Year',
 'Month',
 'Week',
 'Inventory_to_Sales_Ratio',
 'Is_Weekend',
 'Day_of_Week',
 'Day',
 'Promotion_Discount',
 'Price_Difference_Percentage',
 'Price_Difference',
 'Inventory_Gap',
 'Previous_Demand',
 'Previous_Units_Sold',
 'Rolling_7_Day_Demand',
 'Rolling_7_Day_Sales']

##### Important point

For this dataset, Demand is our target:

- Demand → What we are trying to predict

Our lagged features are safe:

- Previous_Demand
- Previous_Units_Sold
- Rolling_7_Day_Demand
- Rolling_7_Day_Sales
because they are calculated using previous days only.

However, we need to think carefully about:

- Units Sold
- Units Ordered
- Inventory Level
These values may or may not be available depending on when the prediction is being made.

For a realistic retail system, we can frame the prediction as:

- Predicting demand for a day using information available at the beginning of that day.

Under that framing, today's inventory, price, promotion, weather forecast, etc. can potentially be available.

But today's Units Sold cannot be used, because sales are the result of today's demand.

So we should remove Units Sold from the model features.

##### Removing Target-Leaking Features

In [14]:
# Removing current-day Units Sold because it is influenced by the demand being predicted.

feature_columns = [
    col for col in feature_columns
    if col != 'Units Sold'
]

In [15]:
# Recreating the feature matrix after removing the current-day sales feature.

X = df[feature_columns]

y = df[target]

In [16]:
# Displaying the final features that will be used for model development.

X.columns.tolist()

['Store ID',
 'Product ID',
 'Category',
 'Region',
 'Inventory Level',
 'Units Ordered',
 'Price',
 'Discount',
 'Weather Condition',
 'Promotion',
 'Competitor Pricing',
 'Seasonality',
 'Epidemic',
 'Year',
 'Month',
 'Week',
 'Inventory_to_Sales_Ratio',
 'Is_Weekend',
 'Day_of_Week',
 'Day',
 'Promotion_Discount',
 'Price_Difference_Percentage',
 'Price_Difference',
 'Inventory_Gap',
 'Previous_Demand',
 'Previous_Units_Sold',
 'Rolling_7_Day_Demand',
 'Rolling_7_Day_Sales']

#### Creating a Time-Based Train-Test Split

This is very important for our project.

We should not randomly split this dataset.

Why?

Imagine:

- 2022 → Training data
- 2023 → Training data
- 2024 → Testing data

This is realistic.

The model learns from the past and is tested on the future.

If we randomly shuffle the data, the model could see information from 2024 while trying to predict 2022, which isn't how real forecasting works.

#### Checking the Date Range

In [17]:
# Checking the available date range before creating the time-based split.

print("Minimum Date:", df['Date'].min())
print("Maximum Date:", df['Date'].max())

Minimum Date: 2022-01-08 00:00:00
Maximum Date: 2024-01-30 00:00:00


#### Defining Split Date

We have data from 2022-01-01 to 2024-01-30.

We'll use the last part of the timeline as our test set.

In [18]:
# Defining the date boundary for separating historical training data from future testing data.

split_date = pd.Timestamp('2023-10-01')

#### Creating Training and Testing Sets

In [19]:
# Creating the training dataset using observations before the split date.

train_data = df[
    df['Date'] < split_date
].copy()


# Creating the testing dataset using observations on and after the split date.

test_data = df[
    df['Date'] >= split_date
].copy()

#### Creating X and y for Training

In [20]:
# Creating the training features and target variable.

X_train = train_data[feature_columns]

y_train = train_data[target]

#### Creating X and y for Testing

In [21]:
# Creating the testing features and target variable.

X_test = test_data[feature_columns]

y_test = test_data[target]

#### Checking Split Sizes

In [22]:
# Checking the number of observations in the training and testing datasets.

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

Training Data Shape: (63100, 28)
Testing Data Shape: (12200, 28)


#### Checking Split Dates

In [23]:
# Checking the date ranges of the training and testing datasets.

print(
    "Training Date Range:",
    train_data['Date'].min(),
    "to",
    train_data['Date'].max()
)

print(
    "Testing Date Range:",
    test_data['Date'].min(),
    "to",
    test_data['Date'].max()
)

Training Date Range: 2022-01-08 00:00:00 to 2023-09-30 00:00:00
Testing Date Range: 2023-10-01 00:00:00 to 2024-01-30 00:00:00


Our data is now conceptually:


                 TIME
                  ↓

2022 ─────────────────────── 2023-09-30 | 2023-10-01 ─────── 2024
       TRAINING DATA                     |       TEST DATA
                                        ↑
                                   Split Date



The model will learn from the past and then we'll ask it to predict the future.



#### Encoding Categorical Features

##### Defining Categorical Features

In [24]:
# Defining the categorical features that will be encoded before model training.

categorical_features = [
    'Store ID',
    'Product ID',
    'Category',
    'Region',
    'Weather Condition',
    'Seasonality'
]

##### Defining Numerical Features

In [25]:
# Defining the numerical features that will be passed directly to the models.

numerical_features = [
    col for col in feature_columns
    if col not in categorical_features
]

##### Displaying Feature Groups

In [26]:
# Displaying the categorical and numerical feature groups for verification.

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

Categorical Features:
['Store ID', 'Product ID', 'Category', 'Region', 'Weather Condition', 'Seasonality']

Numerical Features:
['Inventory Level', 'Units Ordered', 'Price', 'Discount', 'Promotion', 'Competitor Pricing', 'Epidemic', 'Year', 'Month', 'Week', 'Inventory_to_Sales_Ratio', 'Is_Weekend', 'Day_of_Week', 'Day', 'Promotion_Discount', 'Price_Difference_Percentage', 'Price_Difference', 'Inventory_Gap', 'Previous_Demand', 'Previous_Units_Sold', 'Rolling_7_Day_Demand', 'Rolling_7_Day_Sales']


#### Creating the Preprocessing Pipeline

In [27]:
# Creating a one-hot encoder for converting categorical features into numerical features.

encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

##### Creating the Preprocessing Transformer

In [ ]:
# Creating a preprocessing transformer for encoding categorical features while retaining numerical features.

preprocessor = ColumnTransformer(
    transformers=[
        (
            'categorical',
            encoder,
            categorical_features
        ),
        (
            'numerical',
            'passthrough',
            numerical_features
        )
    ]
)

#### Checking the Preprocessing Pipeline

Before training a model, let's make sure the transformation works.

##### Fitting the Preprocessor

In [29]:
# Fitting the preprocessing transformer using only the training data.

preprocessor.fit(X_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``

##### Transforming Training Data

In [30]:
# Transforming the training features using the fitted preprocessing pipeline.

X_train_processed = preprocessor.transform(
    X_train
)

##### Transforming Testing Data

In [31]:
# Transforming the testing features using the preprocessing pipeline fitted on the training data.

X_test_processed = preprocessor.transform(
    X_test
)

##### Checking Processed Data Shape

In [32]:
# Checking the dimensions of the processed training and testing feature matrices.

print("Processed Training Shape:", X_train_processed.shape)
print("Processed Testing Shape:", X_test_processed.shape)

Processed Training Shape: (63100, 64)
Processed Testing Shape: (12200, 64)


#### Establishing a Baseline Model

For a demand forecasting problem, a very useful baseline is:

- Predicting the average historical demand from the training data for every future observation.

It's deliberately simple.

If our baseline MAE is 45 and a Random Forest gives MAE 20, we know the model is providing meaningful improvement.

##### Calculating Baseline Prediction

In [33]:
# Calculating the average demand from the training data for the baseline prediction.

baseline_prediction = y_train.mean()

##### Creating Baseline Predictions

In [34]:
# Creating baseline predictions using the average training demand.

y_baseline_pred = np.full(
    shape=len(y_test),
    fill_value=baseline_prediction
)

##### Evaluating Baseline MAE

In [35]:
# Calculating the Mean Absolute Error for the baseline predictions.

baseline_mae = mean_absolute_error(
    y_test,
    y_baseline_pred
)

print("Baseline MAE:", baseline_mae)

Baseline MAE: 36.491772642955496


##### Evaluating Baseline RMSE

In [36]:
# Calculating the Root Mean Squared Error for the baseline predictions.

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_baseline_pred
    )
)

print("Baseline RMSE:", baseline_rmse)

Baseline RMSE: 45.41942310451735


##### Evaluating Baseline R²

In [37]:
# Calculating the R² score for the baseline predictions.

baseline_r2 = r2_score(
    y_test,
    y_baseline_pred
)

print("Baseline R²:", baseline_r2)

Baseline R²: -0.021844943419516483


##### Creating a Baseline Results Table

In [38]:
# Creating a results table for storing baseline model performance.

model_results = pd.DataFrame({
    'Model': ['Baseline'],
    'MAE': [baseline_mae],
    'RMSE': [baseline_rmse],
    'R2': [baseline_r2]
})

model_results

,Model,MAE,RMSE,R2
0,Baseline,36.491773,45.419423,-0.021845
